# Análisis multitarea simultáneo en español: sentimiento, entidades y tópicos en textos comerciales y jurídicos

**Curso:** Procesamiento de Lenguaje Natural · Unidad 6 — Aplicaciones avanzadas y evaluación
**Institución:** Universidad Tecnológica de Bolívar
**Referencias:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 4 (clasificación y sentimiento), Cap. 10–11 (LLM y *fine-tuning*), Cap. 17 (NER); Studer, S. et al. (2021). *Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology*
**Notebooks relacionados:** `ner_spacy.ipynb` (reglas y NER preentrenado), `bert_classifier.ipynb` (ajuste de BERT y validación cruzada), `text_generation_business.ipynb` (LLM generativos), `metrics_llm_business.ipynb` (F1 y métricas)
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook compara modelos de lenguaje abiertos de **hasta 2 mil millones de parámetros**, ejecutables en CPU, para resolver **tres tareas a la vez sobre el mismo texto en español**: (1) la **polaridad** del texto, el sentimiento de una reseña comercial o el sentido del fallo (*concede*, *niega*, *improcedente*) de una sentencia de tutela, (2) el **tópico** dentro de una lista fija de categorías y (3) las **entidades nombradas** (personas, organizaciones, lugares, fechas y montos). Se enfrentan dos familias de sistemas: **encoders multitarea** (BETO, RoBERTa-BNE y XLM-R con tres cabezas entrenadas juntas) y **LLM generativos pequeños** (Salamandra-2B y Qwen2.5-1.5B) que devuelven las tres respuestas en un único JSON. Todo el proceso sigue las seis fases de **CRISP-ML(Q)**, con controles de calidad en cada una.

**Objetivo:** decidir, con evidencia medida en un conjunto de prueba anotado por personas, qué modelo pequeño conviene para extraer en una sola pasada la información clave de reseñas de clientes y de sentencias, y cuánto cuesta en tiempo de cómputo.

## Ejemplos de Aplicación en el Mundo Real

### Comercio minorista — Tablero de opinión de clientes

- **Problema:** una cadena recibe miles de reseñas y PQRS al mes y necesita saber qué categorías generan quejas, sobre qué marcas o tiendas y por qué montos.
- **Entrada:** texto de cada reseña o PQRS.
- **Cómo se aplica el análisis multitarea:**
  1. Un solo modelo devuelve sentimiento, categoría de producto y entidades (marca, ciudad, monto).
  2. Las salidas alimentan una tabla con una fila por mensaje.
  3. Los mensajes negativos con montos altos se priorizan para servicio al cliente.
- **Salida:** JSON por mensaje y tablero por categoría y sentimiento.
- **Decisión:** dónde actuar primero; los casos dudosos pasan a revisión humana.

### Rama judicial — Clasificación inicial de sentencias de tutela

- **Problema:** la Corte Constitucional recibe más de 2.500 fallos de tutela al día para seleccionar cuáles revisa ([Valora Analitik](https://www.valoraanalitik.com/?p=86661), sobre la herramienta Pretoria).
- **Entrada:** síntesis o texto inicial de cada sentencia.
- **Cómo se aplica el análisis multitarea:**
  1. El modelo identifica el derecho en discusión (salud, pensiones, educación…).
  2. Determina el sentido del fallo (concede, niega, improcedente).
  3. Extrae autoridades, entidades accionadas, fechas y montos.
- **Salida:** ficha estructurada por sentencia.
- **Decisión:** apoyo a la búsqueda y la priorización; la decisión de selección sigue siendo humana.

### Firmas de abogados y áreas legales — Vigilancia de litigios

- **Problema:** una empresa quiere saber en cuántas tutelas aparece como accionada, por qué temas y con qué resultado.
- **Entrada:** sentencias públicas.
- **Cómo se aplica:** el mismo modelo filtra por la entidad (ORG), agrupa por tema y cuenta los fallos concedidos.
- **Salida:** informe periódico de riesgo judicial.
- **Decisión:** dónde reforzar procesos internos (p. ej., autorizaciones de salud negadas).

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter antepón `!`):

```bash
pip install torch transformers huggingface_hub accelerate sentencepiece protobuf
pip install pandas numpy scikit-learn matplotlib openpyxl requests beautifulsoup4 lxml spacy
python -m spacy download es_core_news_md
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---|---|
| `torch`, `transformers` | Encoders multitarea y LLM generativos |
| `huggingface_hub` | Descarga de los corpus MeLiSA y NextProcurement |
| `requests`, `bs4` | Descarga y lectura de sentencias de la relatoría de la Corte Constitucional |
| `spacy` | Pre-anotación de entidades por reglas (como en `ner_spacy.ipynb`) |
| `pandas`, `numpy`, `openpyxl` | Datos, partición y hoja de validación en Excel |
| `scikit-learn` | F1 macro, matrices de confusión y particiones estratificadas |
| `matplotlib` | Gráficos de resultados |

**Hardware de referencia:** CPU con **16 GB de RAM** o más y ~15 GB libres en disco. Los LLM de 2B ocupan ~4.5 GB en `bfloat16`. Con `MODO_RAPIDO = True` el notebook completo tarda del orden de 1–2 horas en un portátil; con `False`, varias horas (la mayor parte es la inferencia de los LLM).

**Modelos comparados** (todos abiertos y sin registro):

| Familia | Modelo | Parámetros | Licencia |
|---|---|---|---|
| Encoder | [BETO](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased) `dccuchile/bert-base-spanish-wwm-cased` | 110 M | CC BY 4.0 |
| Encoder | [RoBERTa-BNE (MarIA)](https://huggingface.co/PlanTL-GOB-ES/roberta-base-bne) `PlanTL-GOB-ES/roberta-base-bne` | 125 M | Apache 2.0 |
| Encoder | [XLM-RoBERTa base](https://huggingface.co/FacebookAI/xlm-roberta-base) `FacebookAI/xlm-roberta-base` | 278 M | MIT |
| LLM | [Salamandra-2B-instruct](https://huggingface.co/BSC-LT/salamandra-2b-instruct) `BSC-LT/salamandra-2b-instruct` | 2.3 B | Apache 2.0 |
| LLM | [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) `Qwen/Qwen2.5-1.5B-Instruct` | 1.5 B | Apache 2.0 |

Opcionales (requieren aceptar su licencia en Hugging Face y un *token*): `meta-llama/Llama-3.2-1B-Instruct` y `google/gemma-2-2b-it`. Se activan con `USAR_MODELOS_CON_TOKEN = True`.

**Corpus:**

| Sector | Corpus | Qué aporta | Licencia |
|---|---|---|---|
| Comercial | [MeLiSA](https://huggingface.co/datasets/lpsc-fiuba/melisa) — reseñas de Mercado Libre en español latinoamericano (incluye Colombia) | Sentimiento (estrellas) y tema (categoría de producto) | «other»; la tarjeta no es clara: **solo uso académico** |
| Comercial | [NextProcurement NER](https://huggingface.co/datasets/BSC-LT/NextProcurement-NER-Spanish-UTE-Company-annotated) (BSC) — licitaciones públicas | Empresas y uniones temporales anotadas (→ `ORG`) | CC BY (la tarjeta indica 2.0 y 4.0) |
| Jurídico | [Relatoría de la Corte Constitucional](https://www.corteconstitucional.gov.co/relatoria/) — sentencias de tutela públicas | Síntesis, temas (descriptores) y parte resolutiva (fallo) | Información pública |

## Marco metodológico: CRISP-ML(Q)

CRISP-ML(Q) organiza un proyecto de aprendizaje automático en seis fases y exige, en cada una, **definir requisitos, identificar riesgos y aplicar controles de calidad (QA)**. Este notebook marca cada fase con 🔷.

| Fase CRISP-ML(Q) | Dónde se aborda | Control de calidad (QA) principal |
|---|---|---|
| 🔷 1. Comprensión del negocio y de los datos | Sección 1 | Criterios de éxito medibles; esquema de etiquetas común a los modelos |
| 🔷 2. Ingeniería de datos | Sección 2 | Licencias, deduplicación, distribución de etiquetas, partición sin fuga, **validación humana** del conjunto de prueba |
| 🔷 3. Ingeniería del modelo | Sección 3 | Mismos datos, misma truncación y mismos hiperparámetros para todos los encoders; mismo *prompt* para todos los LLM |
| 🔷 4. Evaluación | Sección 4 | F1 por tarea, puntaje global, intervalos de confianza *bootstrap*, latencia y análisis de errores |
| 🔷 5. Despliegue | Sección 5 | Función única `analizar(texto, sector)` con salida JSON validada y ficha del modelo |
| 🔷 6. Monitoreo y mantenimiento | Sección 6 | Deriva de la distribución de predicciones, tasa de JSON inválido y plan de re-anotación |

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|---|---|
| **IEEE 7000-2021** — Proceso para abordar consideraciones éticas en el diseño de sistemas | Se declaran los valores en juego (exactitud, equidad entre sectores, privacidad de las partes) y la revisión humana obligatoria en el uso jurídico. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Cada predicción se entrega como JSON legible; la ficha del modelo (sección 5) documenta datos, métricas y límites. |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | Se reportan métricas **por sector y por clase**, no solo un promedio, para detectar clases o dominios desfavorecidos. |
| **IEEE 3119-2025** — Adquisición de IA y sistemas de decisión automatizada | La comparación de modelos con datos y métricas propios es el tipo de evaluación previa que exige adoptar IA de terceros. |
| **ISO/IEC 42001:2023** — Sistema de gestión de IA | Las decisiones de datos, las particiones y los resultados quedan registrados en archivos reproducibles. |
| **ISO/IEC 23894:2023** — Gestión de riesgos de IA | Riesgos documentados: etiquetas automáticas con ruido, licencia dudosa de MeLiSA, sesgo de dominio y salidas JSON inválidas. |
| **OSI · Open Source AI Definition 1.0 (2024)** | Solo se usan modelos con pesos abiertos y licencias permisivas; Salamandra publica además sus datos y código de entrenamiento. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. **Formular** un problema multitarea (polaridad + tópico + entidades) con un esquema de etiquetas común a dos sectores.
2. **Construir** conjuntos de datos a partir de corpus públicos y de páginas web, con etiquetas automáticas por reglas y validación humana.
3. **Entrenar** un encoder con tres cabezas que aprende las tres tareas a la vez, aun cuando cada corpus solo trae algunas etiquetas.
4. **Diseñar** un *prompt* para que un LLM pequeño devuelva las tres respuestas en un JSON válido.
5. **Comparar** modelos con F1 por tarea, intervalos de confianza y costo de cómputo, siguiendo CRISP-ML(Q).

## Decisiones de diseño acordadas

| Tema | Decisión |
|---|---|
| Tareas | Las tres en ambos sectores y **simultáneas** sobre el mismo texto |
| Polaridad | Comercial: sentimiento (negativo, neutro, positivo). Jurídico: sentido del fallo (concede, niega, improcedente) |
| Tópicos | Categorías fijas. Comercial: las 5 categorías de MeLiSA. Jurídico: salud, pensiones y seguridad social, educación, trabajo, petición, debido proceso, otro |
| Entidades | Esquema común: `PER`, `ORG`, `LOC`, `FECHA`, `MONTO` |
| Datos de entrenamiento | MeLiSA y NextProcurement (comercial); sentencias de tutela de la relatoría (jurídico). Entidades por reglas (spaCy + patrones) |
| Conjunto de prueba | ~100 textos por sector con las 3 etiquetas, **validados por personas** en una hoja de Excel |
| Modelos | ≤ 2B parámetros, CPU. Encoders ajustados + LLM con *few-shot* |
| Criterio de selección | Mayor **puntaje global** (promedio de los tres F1) en el conjunto de prueba; si dos modelos no se distinguen estadísticamente, gana el más rápido |

> ℹ️ **Límites conocidos:** las entidades de entrenamiento y el tema jurídico se generan por reglas (etiquetas «plata», con ruido); el fallo se infiere de la parte resolutiva con reglas. Por eso la validación humana del conjunto de prueba es indispensable antes de sacar conclusiones.

---

## 0. Configuración del entorno

In [1]:
# Librerías estándar
import json, math, random, re, time, unicodedata, warnings
from collections import Counter
from pathlib import Path

# Datos y métricas
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Modelos
import torch
from torch import nn

warnings.filterwarnings('ignore', category=UserWarning)

# Descargas de Hugging Face sin compresión brotli: algunas combinaciones de httpx y brotlicffi
# (p. ej. en Anaconda) fallan con «DecodingError: brotli: decoder process called with data…»
try:
    import huggingface_hub
    from huggingface_hub.utils import _http as _hf_http
    _fabrica_hf = _hf_http.default_client_factory

    def _cliente_sin_brotli():
        cliente = _fabrica_hf()
        cliente.headers['Accept-Encoding'] = 'identity'
        return cliente

    huggingface_hub.set_client_factory(_cliente_sin_brotli)
except (ImportError, AttributeError):
    pass   # versiones antiguas de huggingface_hub (basadas en requests) no lo necesitan

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Parámetros generales -------------------------------------------------
MODO_RAPIDO = True              # True: muestras pequeñas para clase; False: experimento completo
USAR_MODELOS_CON_TOKEN = False  # True: añade Llama-3.2-1B y Gemma-2-2B (requieren token de Hugging Face)

DATA_DIR = Path('data/multitarea')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path('results/multitarea')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_RESENAS = 1200 if MODO_RAPIDO else 6000          # reseñas MeLiSA (entrenamiento + prueba)
N_ORACIONES_LICIT = 600 if MODO_RAPIDO else 3000   # oraciones de licitaciones (solo entidades)
N_TUTELAS = 300                                    # sentencias de tutela a descargar
N_PRUEBA = 100                                     # textos del conjunto de prueba conjunto, por sector
N_PRUEBA_LLM = 40 if MODO_RAPIDO else N_PRUEBA     # textos de prueba evaluados con los LLM (son lentos en CPU)
MAX_PALABRAS = 200                                 # misma truncación para todos los modelos

ENCODERS = {
    'BETO': ['dccuchile/bert-base-spanish-wwm-cased'],
    'RoBERTa-BNE': ['PlanTL-GOB-ES/roberta-base-bne', 'BSC-LT/roberta-base-bne'],  # el segundo es respaldo
    'XLM-R': ['FacebookAI/xlm-roberta-base'],
}
LLMS = {
    'Salamandra-2B': 'BSC-LT/salamandra-2b-instruct',
    'Qwen2.5-1.5B': 'Qwen/Qwen2.5-1.5B-Instruct',
}
if USAR_MODELOS_CON_TOKEN:
    LLMS.update({'Llama-3.2-1B': 'meta-llama/Llama-3.2-1B-Instruct',
                 'Gemma-2-2B': 'google/gemma-2-2b-it'})

print(f'torch {torch.__version__} · hilos de CPU: {torch.get_num_threads()} · MODO_RAPIDO={MODO_RAPIDO}')

torch 2.14.0+cpu · hilos de CPU: 4 · MODO_RAPIDO=True


> **Interpretación de la salida:** confirma la versión de PyTorch y cuántos hilos de CPU usará. `MODO_RAPIDO = True` reduce las muestras para que el notebook quepa en una o dos sesiones de clase; las conclusiones finales deben sacarse con `False`.

---

## 1. 🔷 Fase 1 — Comprensión del negocio y de los datos

**Pregunta de negocio:** ¿qué modelo abierto de hasta 2B parámetros, ejecutable en un computador sin GPU, extrae mejor —y en una sola pasada— la polaridad, el tema y las entidades de textos comerciales y jurídicos en español?

**Criterios de éxito (definidos antes de ver resultados):**

| Criterio | Medida | Umbral orientativo |
|---|---|---|
| Calidad por tarea | F1 macro (polaridad y tema); F1 micro por entidad (texto + tipo exactos) | Superar la línea base de la clase mayoritaria |
| Calidad conjunta | Puntaje global = promedio de los tres F1 | El mayor, con IC 95 % que no se solape con el segundo |
| Robustez de formato (LLM) | % de respuestas con JSON válido | ≥ 95 % |
| Costo | Segundos por texto en CPU | Reportar; desempata |

**Riesgos identificados:** etiquetas automáticas con ruido; textos jurídicos largos (se truncan); desbalance de clases (pocas sentencias de educación o petición); licencia no clara de MeLiSA (uso académico).

**QA de la fase:** el esquema de etiquetas es el mismo para todos los modelos y se fija aquí, antes de entrenar.

In [2]:
# Esquema de etiquetas común a todos los modelos
SECTORES = {
    'comercial': {
        'nombre_polaridad': 'sentimiento',
        'polaridad': ['negativo', 'neutro', 'positivo'],
        'temas': ['hogar', 'tecnología y electrónica', 'salud, ropa y cuidado personal',
                  'arte y entretenimiento', 'alimentos y bebidas'],
        'descripcion': 'reseña de un producto comprado en línea',
    },
    'juridico': {
        'nombre_polaridad': 'sentido del fallo',
        'polaridad': ['concede', 'niega', 'improcedente'],
        'temas': ['salud', 'pensiones y seguridad social', 'educación', 'trabajo',
                  'petición', 'debido proceso', 'otro'],
        'descripcion': 'síntesis de una sentencia de tutela de la Corte Constitucional de Colombia',
    },
}
TIPOS_ENT = ['PER', 'ORG', 'LOC', 'FECHA', 'MONTO']
ETIQUETAS_BIO = ['O'] + [f'{p}-{t}' for t in TIPOS_ENT for p in ('B', 'I')]
BIO2ID = {e: i for i, e in enumerate(ETIQUETAS_BIO)}


def sin_tildes(texto: str) -> str:
    """Minúsculas y sin tildes, para comparar textos y etiquetas."""
    t = unicodedata.normalize('NFKD', texto.lower())
    return ''.join(c for c in t if not unicodedata.combining(c))


def truncar(texto: str, max_palabras: int = MAX_PALABRAS) -> str:
    """Primeras max_palabras palabras: la misma entrada para encoders y LLM."""
    palabras = texto.split()
    return ' '.join(palabras[:max_palabras])


for s, cfg in SECTORES.items():
    print(f"{s:10} {cfg['nombre_polaridad']}: {cfg['polaridad']}")
    print(f"{'':10} temas ({len(cfg['temas'])}): {cfg['temas']}")
print('Entidades:', TIPOS_ENT, '→', len(ETIQUETAS_BIO), 'etiquetas BIO')

comercial  sentimiento: ['negativo', 'neutro', 'positivo']
           temas (5): ['hogar', 'tecnología y electrónica', 'salud, ropa y cuidado personal', 'arte y entretenimiento', 'alimentos y bebidas']
juridico   sentido del fallo: ['concede', 'niega', 'improcedente']
           temas (7): ['salud', 'pensiones y seguridad social', 'educación', 'trabajo', 'petición', 'debido proceso', 'otro']
Entidades: ['PER', 'ORG', 'LOC', 'FECHA', 'MONTO'] → 11 etiquetas BIO


> **Interpretación de la salida:** cada texto recibirá tres respuestas: una polaridad (3 clases en ambos sectores), un tema (5 comerciales o 7 jurídicos) y una lista de entidades de 5 tipos. Para el encoder, las entidades se codifican con el esquema **BIO** (`B-` inicio, `I-` continuación, `O` fuera de entidad): 11 etiquetas por *token*.

---

## 2. 🔷 Fase 2 — Ingeniería de datos

Todos los textos se convierten a un **formato único**: un registro por texto con `sector`, `texto`, `polaridad`, `tema` y `entidades` (lista de `{inicio, fin, tipo, texto}`). Si un corpus no trae una etiqueta, el campo queda en `None` y esa tarea no se usa para entrenar con ese texto.

### 2.1 Reseñas comerciales: MeLiSA

Se descargan los CSV del repositorio de Hugging Face, se descartan las reseñas en portugués (Brasil) y se toma una muestra estratificada por estrellas y categoría.

**Decisión de etiquetado del sentimiento:** 1–2 estrellas → negativo, 3 → neutro, 4–5 → positivo (regla habitual en reseñas; editable en `ESTRELLAS_A_SENTIMIENTO`).

In [3]:
from huggingface_hub import snapshot_download
import requests

ESTRELLAS_A_SENTIMIENTO = {1: 'negativo', 2: 'negativo', 3: 'neutro', 4: 'positivo', 5: 'positivo'}


def descargar_repo(repo_id: str, carpeta: Path) -> Path:
    """Descarga un repositorio de datos de Hugging Face sin compresión brotli.

    Algunas combinaciones de httpx y brotlicffi (p. ej. en Anaconda) fallan con
    'DecodingError: brotli: decoder process called with data...'. Pedir la respuesta
    sin comprimir ('Accept-Encoding: identity') evita el error.
    """
    return Path(snapshot_download(repo_id, repo_type='dataset', local_dir=carpeta,
                                  headers={'Accept-Encoding': 'identity'}))


def normalizar_categoria(cat: str) -> str:
    """Lleva la categoría de MeLiSA (p. ej. 'Hogar / Casa') a una de las 5 etiquetas en español."""
    c = sin_tildes(str(cat))
    for clave, tema in [('hogar', 'hogar'), ('tecnolog', 'tecnología y electrónica'),
                        ('salud', 'salud, ropa y cuidado personal'), ('arte', 'arte y entretenimiento'),
                        ('aliment', 'alimentos y bebidas')]:
        if clave in c:
            return tema
    return None


def cargar_melisa() -> pd.DataFrame:
    """Descarga MeLiSA y devuelve las reseñas en español con sus columnas originales."""
    destino = descargar_repo('lpsc-fiuba/melisa', DATA_DIR / 'melisa')
    csvs = sorted(destino.rglob('*.csv'))
    if not csvs:
        # Algunas versiones guardan los CSV fuera del repositorio y los enlazan desde el script de carga
        urls = set()
        for script in destino.rglob('*.py'):
            urls |= set(re.findall(r'https?://[^\s\'"]+\.csv', script.read_text(encoding='utf-8')))
        for url in sorted(urls):
            archivo = destino / Path(url).name
            if not archivo.exists():
                archivo.write_bytes(requests.get(url, timeout=120).content)
        csvs = sorted(destino.rglob('*.csv'))
    if not csvs:
        raise FileNotFoundError(f'No se encontraron CSV de MeLiSA en {destino}: {list(destino.rglob("*"))[:20]}')
    df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    df = df[df['country'] != 'MLB']            # MLB = Brasil (portugués)
    return df.drop_duplicates(subset='review_content')


melisa = cargar_melisa()
melisa['polaridad'] = melisa['review_rate'].map(ESTRELLAS_A_SENTIMIENTO)
melisa['tema'] = melisa['category'].map(normalizar_categoria)
melisa = melisa.dropna(subset=['polaridad', 'tema', 'review_content'])
melisa['texto'] = (melisa['review_title'].fillna('').str.strip() + '. ' +
                   melisa['review_content'].str.strip()).str.strip('. ').map(truncar)
melisa = melisa[melisa['texto'].str.split().str.len() >= 8]   # descarta reseñas de una o dos palabras

# Muestra estratificada por sentimiento × categoría
n_por_celda = max(1, N_RESENAS // (3 * 5))
muestra_com = pd.concat([g.sample(min(len(g), n_por_celda), random_state=SEED)
                         for _, g in melisa.groupby(['polaridad', 'tema'])])
print(f'Reseñas en español disponibles: {len(melisa):,} · muestra: {len(muestra_com):,}')
print(muestra_com['country'].value_counts().to_string())
pd.crosstab(muestra_com['tema'], muestra_com['polaridad'])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Reseñas en español disponibles: 10 · muestra: 10
country
MLA    9
MLM    1


polaridad,negativo
tema,
alimentos y bebidas,7
"salud, ropa y cuidado personal",3


> **Interpretación de la salida:** la tabla cruzada debe mostrar celdas parecidas (muestreo estratificado); así ninguna combinación de sentimiento y categoría domina el entrenamiento. Revisa también los países: la muestra mezcla variedades del español latinoamericano (MCO = Colombia). **QA:** se eliminaron duplicados exactos y reseñas muy cortas.

### 2.2 Entidades comerciales: licitaciones públicas (NextProcurement)

Este corpus del BSC anota empresas (`SINGLE_COMPANY`) y uniones temporales (`UTE`) en documentos de contratación pública. Ambas se convierten a `ORG`. Solo aporta **entidades**: polaridad y tema quedan en `None`.

In [4]:
def leer_iob(lineas: list) -> list:
    """Convierte líneas 'token<TAB>etiqueta' en oraciones [(tokens, etiquetas)]; línea vacía = fin de oración."""
    oraciones, toks, tags = [], [], []
    for ln in lineas:
        ln = ln.rstrip('\n')
        if not ln.strip():
            if toks:
                oraciones.append((toks, tags)); toks, tags = [], []
            continue
        partes = ln.split('\t') if '\t' in ln else ln.rsplit(' ', 1)
        if len(partes) < 2:
            continue
        toks.append(partes[0]); tags.append(partes[-1].strip())
        if partes[0] in {'.', ';'} and len(toks) > 40:       # corta oraciones muy largas
            oraciones.append((toks, tags)); toks, tags = [], []
    if toks:
        oraciones.append((toks, tags))
    return oraciones


def iob_a_registro(tokens: list, tags: list, mapa: dict) -> dict:
    """Une los tokens con espacios y convierte las etiquetas IOB en entidades con posiciones de carácter."""
    texto, ents, pos, actual = '', [], 0, None
    for tok, tag in zip(tokens, tags):
        inicio = len(texto) + (1 if texto else 0)
        texto = f'{texto} {tok}' if texto else tok
        fin = len(texto)
        tipo = mapa.get(tag[2:]) if tag[:2] in ('B-', 'I-') else None
        if tipo and (tag.startswith('B-') or actual is None or actual['tipo'] != tipo):
            actual = {'inicio': inicio, 'fin': fin, 'tipo': tipo}; ents.append(actual)
        elif tipo:
            actual['fin'] = fin
        else:
            actual = None
    for e in ents:
        e['texto'] = texto[e['inicio']:e['fin']]
    return {'texto': texto, 'entidades': ents}


def cargar_licitaciones() -> list:
    destino = descargar_repo('BSC-LT/NextProcurement-NER-Spanish-UTE-Company-annotated',
                             DATA_DIR / 'nextprocurement')
    oraciones = []
    for f in sorted(destino.rglob('*')):
        if f.suffix == '.parquet':
            d = pd.read_parquet(f)
            if {'tokens', 'ner_tags'} <= set(d.columns):
                oraciones += [(list(t), list(g)) for t, g in zip(d['tokens'], d['ner_tags'])]
            else:   # una columna de texto con 'token\tetiqueta' por fila
                oraciones += leer_iob(d.iloc[:, 0].astype(str).tolist())
        elif f.suffix in {'.txt', '.conll', '.tsv', '.iob', '.iob2'} and f.stat().st_size > 1000:
            oraciones += leer_iob(f.read_text(encoding='utf-8').splitlines())
    if not oraciones:
        raise FileNotFoundError(f'No se reconoció el formato de los archivos en {destino}')
    return oraciones


MAPA_LICIT = {'SINGLE_COMPANY': 'ORG', 'UTE': 'ORG'}
oraciones_licit = cargar_licitaciones()
con_ent = [o for o in oraciones_licit if any(t != 'O' for t in o[1])]
sin_ent = [o for o in oraciones_licit if all(t == 'O' for t in o[1])]
random.Random(SEED).shuffle(con_ent); random.Random(SEED).shuffle(sin_ent)
# 80 % oraciones con empresas y 20 % sin entidades, para que el modelo también aprenda la clase O
seleccion = con_ent[:int(N_ORACIONES_LICIT * 0.8)] + sin_ent[:int(N_ORACIONES_LICIT * 0.2)]
licitaciones = [dict(iob_a_registro(t, g, MAPA_LICIT), sector='comercial', polaridad=None, tema=None,
                     fuente='nextprocurement') for t, g in seleccion]
print(f'Oraciones leídas: {len(oraciones_licit):,} · con empresas: {len(con_ent):,} · seleccionadas: {len(licitaciones)}')
print('Ejemplo:', licitaciones[0]['texto'][:160], '\n →', [(e['texto'], e['tipo']) for e in licitaciones[0]['entidades']])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Oraciones leídas: 8,441 · con empresas: 821 · seleccionadas: 600
Ejemplo: | \' _ ] LICITADOR : A48010573 _— T * Razón social : OBRASCON HUARTE LAIN SA T * Fecha de presentación : 12 / 04 / 2019 12:22:25 * Estado del licitador : * Valo 
 → [('OBRASCON HUARTE LAIN SA', 'ORG')]


> **Interpretación de la salida:** cada oración queda con sus empresas marcadas como `ORG`. Este corpus no anota personas, lugares, fechas ni montos: si aparecen, quedan como `O`, lo que puede enseñarle al modelo a ignorarlos. Por eso solo se usa una parte y se combina con las reseñas pre-anotadas por reglas (sección 2.4).

### 2.3 Sentencias de tutela: relatoría de la Corte Constitucional

Se descargan sentencias de tutela recientes (2021–2025), que incluyen una **síntesis de la decisión**. De cada una se obtiene:

| Campo | Origen | Cómo |
|---|---|---|
| `texto` | Sección *Síntesis de la decisión* | Truncada a `MAX_PALABRAS` |
| `tema` | Descriptores en mayúscula al inicio (p. ej. *DERECHO A LA SALUD-…*) | Palabras clave → uno de los 7 temas |
| `polaridad` (fallo) | Parte resolutiva (*RESUELVE*) | Reglas: amparar/tutelar/conceder → concede; improcedente → improcedente; negar/denegar → niega |

Las páginas se guardan en `data/multitarea/tutelas_html/` para no descargarlas de nuevo, y se espera 1 segundo entre peticiones para no cargar el servidor público.

In [ ]:
from bs4 import BeautifulSoup

CACHE_HTML = DATA_DIR / 'tutelas_html'
CACHE_HTML.mkdir(exist_ok=True)
CABECERAS = {'User-Agent': 'courseNLP-UTB/1.0 (uso academico)'}


def descargar_sentencia(anio: int, numero: int):
    """Devuelve el HTML de la sentencia T-numero/anio o None si no existe. Usa caché local."""
    aa = str(anio)[2:]
    cache = CACHE_HTML / f'T-{numero:03d}-{aa}.html'
    if cache.exists():
        return cache.read_text(encoding='utf-8') or None
    html = None
    for pref in ('t', 'T'):
        url = f'https://www.corteconstitucional.gov.co/relatoria/{anio}/{pref}-{numero:03d}-{aa}.htm'
        try:
            r = requests.get(url, headers=CABECERAS, timeout=30)
        except requests.RequestException:
            continue
        time.sleep(1.0)
        if r.status_code == 200 and len(r.content) > 5000:
            r.encoding = r.apparent_encoding or 'utf-8'
            html = r.text
            break
    cache.write_text(html or '', encoding='utf-8')
    return html


def texto_plano(html: str) -> str:
    soup = BeautifulSoup(html, 'lxml')
    for s in soup(['script', 'style']):
        s.decompose()
    texto = soup.get_text('\n')
    return re.sub(r'[ \t\xa0]+', ' ', texto)


def extraer_sintesis(texto: str):
    m = re.search(r'S[íi]ntesis de la decisi[óo]n', texto, flags=re.I)
    if not m:
        return None
    resto = texto[m.end():]
    fin = re.search(r'\n\s*(?:SENTENCIA|I\.\s*ANTECEDENTES|ANTECEDENTES)\s*\n', resto)
    sintesis = resto[:fin.start()] if fin else resto[:4000]
    sintesis = re.sub(r'\s+', ' ', sintesis).strip(' .:-')
    return sintesis if len(sintesis) >= 300 else None


def extraer_descriptores(texto: str) -> list:
    """Líneas en mayúscula con guion al inicio del documento (índice temático de la relatoría)."""
    cabecera = re.split(r'S[íi]ntesis de la decisi[óo]n', texto, maxsplit=1, flags=re.I)[0]
    lineas = [ln.strip() for ln in cabecera.split('\n')]
    return [ln for ln in lineas if re.match(r'^[A-ZÁÉÍÓÚÑÜ][A-ZÁÉÍÓÚÑÜ ,]{5,}-', ln)]


CLAVES_TEMA = [
    ('salud', ['salud', 'eps ', 'medicamento', 'tratamiento medico']),
    ('pensiones y seguridad social', ['pension', 'seguridad social', 'colpensiones', 'incapacidad']),
    ('educación', ['educacion', 'estudiante', 'universidad', 'colegio']),
    ('trabajo', ['trabajo', 'laboral', 'estabilidad reforzada', 'despido', 'salario']),
    ('petición', ['derecho de peticion']),
    ('debido proceso', ['debido proceso', 'providencias judiciales', 'defecto']),
]


def tema_desde_descriptores(descriptores: list) -> str:
    conteo = Counter()
    for d in descriptores:
        d_n = sin_tildes(d) + ' '
        for tema, claves in CLAVES_TEMA:
            if any(c in d_n for c in claves):
                conteo[tema] += 1
    return conteo.most_common(1)[0][0] if conteo else 'otro'


def fallo_desde_resuelve(texto: str):
    """Sentido del fallo según la parte resolutiva; None si las reglas no deciden (va a revisión)."""
    pos = [m.start() for m in re.finditer(r'R\s*E\s*S\s*U\s*E\s*L\s*V\s*E', texto)]
    if not pos:
        return None
    parte = sin_tildes(texto[pos[-1]:pos[-1] + 3000])
    if re.search(r'\b(amparar|tutelar|conceder|proteger)\b', parte):
        return 'concede'
    if 'improcedente' in parte:
        return 'improcedente'
    if re.search(r'\b(negar|denegar)\b|que (nego|denego)', parte):
        return 'niega'
    return None   # p. ej. carencia actual de objeto: se excluye del entrenamiento


def recolectar_tutelas(n_objetivo: int = N_TUTELAS, anios=(2021, 2022, 2023, 2024, 2025)) -> list:
    rng = random.Random(SEED)
    candidatos = [(a, n) for a in anios for n in range(1, 451)]
    rng.shuffle(candidatos)
    registros, intentos, vistos = [], 0, set()
    for anio, num in candidatos:
        if len(registros) >= n_objetivo:
            break
        intentos += 1
        html = descargar_sentencia(anio, num)
        if not html:
            continue
        plano = texto_plano(html)
        sintesis = extraer_sintesis(plano)
        if not sintesis or sin_tildes(truncar(sintesis)) in vistos:   # QA: sin duplicados
            continue
        vistos.add(sin_tildes(truncar(sintesis)))
        registros.append({'id': f'T-{num:03d}-{str(anio)[2:]}', 'sector': 'juridico', 'fuente': 'relatoria_cc',
                          'texto': truncar(sintesis), 'descriptores': extraer_descriptores(plano),
                          'tema': tema_desde_descriptores(extraer_descriptores(plano)),
                          'polaridad': fallo_desde_resuelve(plano)})
        if len(registros) % 25 == 0:
            print(f'  {len(registros)} sentencias útiles de {intentos} intentos')
    return registros


tutelas = recolectar_tutelas()
df_tut = pd.DataFrame(tutelas)
print(f'Sentencias con síntesis: {len(df_tut)}')
print('Fallo (reglas):', df_tut['polaridad'].value_counts(dropna=False).to_dict())
print('Tema (descriptores):', df_tut['tema'].value_counts().to_dict())

> **Interpretación de la salida:** la primera ejecución tarda (una petición por segundo); las siguientes leen la caché. Revisa dos cosas: (1) cuántas sentencias quedaron con fallo `None` —las reglas no decidieron, por ejemplo por *carencia actual de objeto*—; se excluyen del entrenamiento de la polaridad pero conservan tema y entidades; (2) el desbalance de temas: si `educación` o `petición` tienen muy pocos casos, su F1 será inestable. **QA:** estas etiquetas son automáticas; en la sección 2.6 se validan a mano las del conjunto de prueba.

### 2.4 Entidades por reglas: spaCy + patrones

Como en `ner_spacy.ipynb`, el modelo preentrenado de spaCy encuentra personas, organizaciones y lugares, pero no fechas ni montos, y confunde empresas nuevas con lugares. Se combinan tres fuentes, en este orden de prioridad:

1. **Patrones** para `FECHA` (*12 de marzo de 2024*), `MONTO` (*$2.300.000*, *80.000 millones de pesos*, *3 SMLMV*) y organizaciones con palabra clave (*Nueva EPS*, *Juzgado Tercero…*, *Ministerio de…*).
2. **spaCy** `es_core_news_md` para `PER`, `ORG` y `LOC` que no se solapen con los patrones (se descarta `MISC`).
3. Limpieza: se eliminan entidades que son palabras vacías o empiezan en minúscula.

In [ ]:
import spacy

nlp = spacy.load('es_core_news_md', disable=['lemmatizer'])

MESES = 'enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre'
RE_FECHA = re.compile(rf'\b\d{{1,2}}\s+de\s+(?:{MESES})(?:\s+(?:de|del)\s+\d{{4}})?\b|\b(?:{MESES})\s+de\s+\d{{4}}\b'
                      rf'|\b\d{{1,2}}/\d{{1,2}}/\d{{2,4}}\b', re.I)
RE_MONTO = re.compile(
    r'(?:\$|COP\s?|USD\s?|US\$)\s?\d[\d.,]*(?:\s?(?:mil millones|millones|billones|mil))?(?:\s+de\s+pesos)?'
    r'|\b\d[\d.,]*\s?(?:mil millones de pesos|millones de pesos|billones de pesos|millones|pesos|d[óo]lares|'
    r'salarios? m[íi]nimos?(?:\s+legales)?(?:\s+mensuales)?(?:\s+vigentes)?|SMLMV|smlmv)\b', re.I)
CLAVES_ORG = (r'Nueva EPS|EPS|IPS|Juzgado|Tribunal|Corte|Ministerio|Superintendencia|Fiscal[íi]a|Procuradur[íi]a|'
              r'Defensor[íi]a|Personer[íi]a|Alcald[íi]a|Gobernaci[óo]n|Colpensiones|Universidad|Secretar[íi]a|'
              r'Instituto|Unidad|Consejo|Registradur[íi]a|Polic[íi]a|Ej[ée]rcito|Banco|Fondo')
RE_ORG = re.compile(rf'\b(?:{CLAVES_ORG})\b(?:\s+(?:de|del|la|las|los|y|para|el)?\s*'
                    rf'(?:[A-ZÁÉÍÓÚÑ][\wáéíóúñ.\-]*|\d+[°º]?))*')
CONECTORES = {'de', 'del', 'la', 'las', 'los', 'y', 'para', 'el'}


def _recortar(texto: str, ini: int, fin: int):
    """Quita conectores sueltos al final de una entidad encontrada por patrón."""
    palabras = texto[ini:fin].split()
    while palabras and palabras[-1].lower() in CONECTORES:
        palabras.pop()
    return ini, ini + len(' '.join(palabras))


def entidades_por_reglas(texto: str) -> list:
    spans = []
    for regex, tipo in ((RE_MONTO, 'MONTO'), (RE_FECHA, 'FECHA'), (RE_ORG, 'ORG')):
        for m in regex.finditer(texto):
            ini, fin = _recortar(texto, m.start(), m.end())
            if fin > ini and not any(ini < s['fin'] and s['inicio'] < fin for s in spans):
                spans.append({'inicio': ini, 'fin': fin, 'tipo': tipo})
    for ent in nlp(texto).ents:
        if ent.label_ not in {'PER', 'ORG', 'LOC'}:
            continue
        if ent.text.lower() in nlp.Defaults.stop_words or not ent.text[0].isupper():
            continue
        if any(ent.start_char < s['fin'] and s['inicio'] < ent.end_char for s in spans):
            continue
        spans.append({'inicio': ent.start_char, 'fin': ent.end_char, 'tipo': ent.label_})
    for s in spans:
        s['texto'] = texto[s['inicio']:s['fin']]
    return sorted(spans, key=lambda s: s['inicio'])


ejemplo = ('La Nueva EPS negó el 12 de marzo de 2024 un tratamiento de $2.300.000 a la señora María Pérez, '
           'residente en Cartagena; el Juzgado Tercero Civil de Cartagena ordenó pagar 3 SMLMV.')
for e in entidades_por_reglas(ejemplo):
    print(f"  {e['tipo']:6} {e['texto']}")

> **Interpretación de la salida:** el ejemplo mezcla las cinco clases. Comprueba que las fechas y los montos salen completos y que *Juzgado Tercero Civil de Cartagena* queda como una sola `ORG`. Estas reglas producen etiquetas **plata** (aproximadas): sirven para entrenar, pero el conjunto de prueba se corrige a mano.

In [ ]:
# Registros comerciales (reseñas) y jurídicos (sentencias), con entidades por reglas
resenas = [{'id': f'MELISA-{i}', 'sector': 'comercial', 'fuente': 'melisa', 'texto': r.texto,
            'polaridad': r.polaridad, 'tema': r.tema} for i, r in enumerate(muestra_com.itertuples())]
for i, r in enumerate(licitaciones):
    r['id'] = f'LICIT-{i}'

t0 = time.time()
for r in resenas + tutelas:
    r['entidades'] = entidades_por_reglas(r['texto'])
print(f'Pre-anotación por reglas: {len(resenas) + len(tutelas)} textos en {time.time() - t0:.0f} s')

for nombre, regs in [('Reseñas', resenas), ('Sentencias', tutelas)]:
    c = Counter(e['tipo'] for r in regs for e in r['entidades'])
    print(f'{nombre:11} entidades por texto: {sum(c.values()) / len(regs):.2f} · {dict(c)}')

> **Interpretación de la salida:** las sentencias suelen tener muchas más entidades por texto (autoridades, fechas, normas citadas como organizaciones) que las reseñas, donde predominan marcas y ciudades. Esa diferencia es en sí un hallazgo: la tarea de entidades es más exigente en el sector jurídico.

### 2.5 Partición sin fuga de información

Por sector se separan **`N_PRUEBA` textos de prueba** que tienen las tres etiquetas (reseñas y sentencias con fallo decidido). El resto es entrenamiento. Las oraciones de licitaciones solo van a entrenamiento porque no tienen polaridad ni tema.

**QA:** no puede haber textos repetidos entre entrenamiento y prueba, y los ejemplos *few-shot* de los LLM se toman solo del entrenamiento.

In [ ]:
def particionar(registros: list, n_prueba: int):
    completos = [r for r in registros if r['polaridad'] and r['tema']]
    estratos = [f"{r['polaridad']}|{r['tema']}" for r in completos]
    # Si un estrato es muy pequeño, se estratifica solo por polaridad
    conteo = Counter(estratos)
    estratos = [e if conteo[e] >= 2 else e.split('|')[0] for e in estratos]
    if min(Counter(estratos).values()) < 2:      # una polaridad con un solo texto: sin estratificar
        estratos = None
    n_prueba = min(n_prueba, len(completos) // 2)
    train_c, test = train_test_split(completos, test_size=n_prueba, random_state=SEED, stratify=estratos)
    ids_test = {r['id'] for r in test}
    train = [r for r in registros if r['id'] not in ids_test]
    return train, test


train_com, test_com = particionar(resenas, N_PRUEBA)
train_com += licitaciones
train_jur, test_jur = particionar(tutelas, N_PRUEBA)

# QA: ningún texto de prueba aparece en entrenamiento
for nombre, tr, te in [('comercial', train_com, test_com), ('juridico', train_jur, test_jur)]:
    fuga = {sin_tildes(r['texto']) for r in tr} & {sin_tildes(r['texto']) for r in te}
    assert not fuga, f'Fuga en {nombre}: {len(fuga)} textos repetidos'
    print(f'{nombre:10} entrenamiento={len(tr):5} prueba={len(te):4} · '
          f"polaridad prueba={dict(Counter(r['polaridad'] for r in te))}")

> **Interpretación de la salida:** el `assert` confirma que no hay textos repetidos entre entrenamiento y prueba. En el sector jurídico la prueba puede quedar desbalanceada (los fallos *concede* suelen ser mayoría en revisión); por eso se usa **F1 macro**, que pesa igual cada clase.

### 2.6 Validación humana del conjunto de prueba

Se exporta una hoja de Excel con las pre-anotaciones. Para cada texto, quien valida corrige las columnas `polaridad_validada`, `tema_validado` y `entidades_validadas` (formato `TIPO: texto | TIPO: texto`) y marca `revisado = si`. Al volver a ejecutar la celda siguiente, el notebook lee la hoja y usa las etiquetas corregidas.

**Instrucciones para quien valida:**
- Polaridad y tema: escoge **un** valor de la lista de la hoja `etiquetas`.
- Entidades: copia el texto **exacto** como aparece; usa solo `PER`, `ORG`, `LOC`, `FECHA`, `MONTO`.
- Si dudas, deja el valor pre-anotado y escribe la duda en `comentario`.

In [ ]:
HOJA = RESULTS_DIR / 'conjunto_prueba_validacion.xlsx'


def ents_a_texto(ents: list) -> str:
    return ' | '.join(f"{e['tipo']}: {e['texto']}" for e in ents)


def texto_a_ents(cadena: str) -> list:
    ents = []
    for parte in str(cadena or '').split('|'):
        if ':' in parte:
            tipo, txt = parte.split(':', 1)
            if tipo.strip().upper() in TIPOS_ENT and txt.strip():
                ents.append({'tipo': tipo.strip().upper(), 'texto': txt.strip()})
    return ents


def exportar_hoja(tests: dict):
    filas = []
    for sector, regs in tests.items():
        for r in regs:
            filas.append({'id': r['id'], 'sector': sector, 'texto': r['texto'],
                          'polaridad_pre': r['polaridad'], 'tema_pre': r['tema'],
                          'entidades_pre': ents_a_texto(r['entidades']),
                          'polaridad_validada': r['polaridad'], 'tema_validado': r['tema'],
                          'entidades_validadas': ents_a_texto(r['entidades']),
                          'revisado': 'no', 'comentario': ''})
    etiquetas = pd.DataFrame({k: pd.Series(v) for k, v in {
        'polaridad_comercial': SECTORES['comercial']['polaridad'], 'tema_comercial': SECTORES['comercial']['temas'],
        'polaridad_juridico': SECTORES['juridico']['polaridad'], 'tema_juridico': SECTORES['juridico']['temas'],
        'tipos_entidad': TIPOS_ENT}.items()})
    with pd.ExcelWriter(HOJA) as w:
        pd.DataFrame(filas).to_excel(w, sheet_name='validacion', index=False)
        etiquetas.to_excel(w, sheet_name='etiquetas', index=False)


def importar_hoja(tests: dict) -> dict:
    hoja = pd.read_excel(HOJA, sheet_name='validacion').fillna('')
    por_id = hoja.set_index('id').to_dict('index')
    revisados = 0
    for sector, regs in tests.items():
        for r in regs:
            f = por_id.get(r['id'])
            if not f:
                continue
            r['polaridad'] = f['polaridad_validada'] or r['polaridad']
            r['tema'] = f['tema_validado'] or r['tema']
            r['entidades'] = texto_a_ents(f['entidades_validadas'])
            r['validado'] = str(f['revisado']).strip().lower() in {'si', 'sí', 'x', '1'}
            revisados += r['validado']
    return revisados


tests = {'comercial': test_com, 'juridico': test_jur}
if not HOJA.exists():
    exportar_hoja(tests)
    print(f'Hoja creada: {HOJA}. Corrígela y vuelve a ejecutar esta celda.')
n_rev = importar_hoja(tests)
total = sum(len(v) for v in tests.values())
print(f'Textos de prueba revisados por personas: {n_rev} de {total}')
if n_rev < total:
    print('⚠️ Los resultados con textos sin revisar se basan en etiquetas automáticas: tómalos como provisionales.')

> **Interpretación de la salida:** la primera vez se crea la hoja y todos los textos figuran como no revisados. Los resultados de la sección 4 solo son concluyentes cuando el contador llega al total. Las entidades validadas se guardan como `TIPO: texto`; la evaluación compara texto y tipo, sin posiciones, igual que la sección 10 de `ner_spacy.ipynb`.

---

## 3. 🔷 Fase 3 — Ingeniería del modelo

Se comparan dos formas de resolver las tres tareas **en una sola pasada**:

| Sistema | Cómo resuelve las 3 tareas a la vez | Entrenamiento |
|---|---|---|
| **Encoder multitarea** | Un encoder compartido y tres cabezas: polaridad y tema sobre el vector `[CLS]`; entidades sobre cada *token* (BIO) | Ajuste supervisado; un modelo por sector |
| **LLM con un solo *prompt*** | El modelo lee el texto y escribe un JSON con `polaridad`, `tema` y `entidades` | Ninguno: instrucciones + 2 ejemplos (*few-shot*) |

### 3.1 Encoder multitarea

**Etiquetas incompletas:** las reseñas tienen las tres etiquetas, pero las licitaciones solo tienen entidades. Cada cabeza calcula su pérdida únicamente con los textos que tienen esa etiqueta (las demás se marcan con `-100`, que PyTorch ignora). La pérdida total es la suma de las tres.

$$\mathcal{L} = \mathcal{L}_{\text{polaridad}} + \mathcal{L}_{\text{tema}} + \mathcal{L}_{\text{entidades}}$$

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

MAX_TOKENS = 256
EPOCAS = 2 if MODO_RAPIDO else 4
LOTE = 16
LR = 3e-5


class EncoderMultitarea(nn.Module):
    """Encoder compartido con tres cabezas: polaridad, tema (texto completo) y entidades (por token)."""

    def __init__(self, nombre_base: str, n_polaridad: int, n_temas: int, n_bio: int = len(ETIQUETAS_BIO)):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(nombre_base)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.cabeza_polaridad = nn.Linear(h, n_polaridad)
        self.cabeza_tema = nn.Linear(h, n_temas)
        self.cabeza_entidades = nn.Linear(h, n_bio)

    def forward(self, input_ids, attention_mask):
        estados = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        cls = self.dropout(estados[:, 0])
        return (self.cabeza_polaridad(cls), self.cabeza_tema(cls),
                self.cabeza_entidades(self.dropout(estados)))


def etiquetas_bio(offsets, entidades) -> list:
    """Etiqueta BIO de cada token a partir de las entidades en posiciones de carácter."""
    tags = []
    for ini, fin in offsets:
        if ini == fin:                       # token especial ([CLS], [SEP], relleno)
            tags.append(-100); continue
        tag = 'O'
        for e in entidades:
            if e['inicio'] <= ini < e['fin']:
                tag = ('B-' if ini == e['inicio'] or not tags or tags[-1] in (-100, BIO2ID['O'])
                       else 'I-') + e['tipo']
                break
        tags.append(BIO2ID[tag])
    return tags


def codificar(registros: list, tok, sector: str) -> dict:
    cfg = SECTORES[sector]
    enc = tok([r['texto'] for r in registros], truncation=True, max_length=MAX_TOKENS,
              padding='max_length', return_offsets_mapping=True, return_tensors='pt')
    pol = [cfg['polaridad'].index(r['polaridad']) if r.get('polaridad') else -100 for r in registros]
    tem = [cfg['temas'].index(r['tema']) if r.get('tema') else -100 for r in registros]
    bio = [etiquetas_bio(off.tolist(), r['entidades']) if r.get('entidades') is not None
           else [-100] * MAX_TOKENS for off, r in zip(enc['offset_mapping'], registros)]
    return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'],
            'offsets': enc['offset_mapping'], 'pol': torch.tensor(pol), 'tema': torch.tensor(tem),
            'bio': torch.tensor(bio)}


def perdida_enmascarada(logits, etiquetas):
    """Entropía cruzada que ignora -100; devuelve 0 si el lote no tiene ninguna etiqueta de esa tarea."""
    if (etiquetas != -100).sum() == 0:
        return logits.sum() * 0.0
    return nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), etiquetas.reshape(-1),
                                       ignore_index=-100)


def cargar_tokenizer_encoder(candidatos: list):
    """Prueba los identificadores en orden (útil si un repositorio cambió de organización)."""
    for nombre in candidatos:
        try:
            return nombre, AutoTokenizer.from_pretrained(nombre)
        except Exception as e:  # noqa: BLE001
            print(f'  no disponible: {nombre} ({type(e).__name__})')
    raise RuntimeError(f'Ninguno disponible: {candidatos}')


def entrenar_encoder(candidatos: list, sector: str, train: list):
    nombre, tok = cargar_tokenizer_encoder(candidatos)
    cfg = SECTORES[sector]
    torch.manual_seed(SEED)
    modelo = EncoderMultitarea(nombre, len(cfg['polaridad']), len(cfg['temas']))
    datos = codificar(train, tok, sector)
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=0.01)
    n = len(train)
    modelo.train()
    for epoca in range(EPOCAS):
        orden = torch.randperm(n, generator=torch.Generator().manual_seed(SEED + epoca))
        total, t0 = 0.0, time.time()
        for i in range(0, n, LOTE):
            idx = orden[i:i + LOTE]
            lp, lt, le = modelo(datos['input_ids'][idx], datos['attention_mask'][idx])
            loss = (perdida_enmascarada(lp, datos['pol'][idx]) + perdida_enmascarada(lt, datos['tema'][idx])
                    + perdida_enmascarada(le, datos['bio'][idx]))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(idx)
        print(f'    época {epoca + 1}/{EPOCAS}: pérdida={total / n:.3f} ({time.time() - t0:.0f} s)')
    modelo.eval()
    return modelo, tok, nombre


def decodificar_bio(ids_bio, offsets, texto) -> list:
    ents, actual = [], None
    for etiqueta, (ini, fin) in zip(ids_bio, offsets):
        if ini == fin:
            continue
        tag = ETIQUETAS_BIO[etiqueta]
        if tag == 'O':
            actual = None; continue
        pref, tipo = tag.split('-', 1)
        if pref == 'B' or actual is None or actual['tipo'] != tipo:
            actual = {'inicio': ini, 'fin': fin, 'tipo': tipo}; ents.append(actual)
        else:
            actual['fin'] = fin
    return [{'tipo': e['tipo'], 'texto': texto[e['inicio']:e['fin']].strip()} for e in ents]


@torch.no_grad()
def predecir_encoder(modelo, tok, sector: str, registros: list) -> list:
    cfg = SECTORES[sector]
    salidas = []
    for i in range(0, len(registros), LOTE):
        lote = registros[i:i + LOTE]
        t0 = time.time()
        enc = tok([r['texto'] for r in lote], truncation=True, max_length=MAX_TOKENS, padding=True,
                  return_offsets_mapping=True, return_tensors='pt')
        lp, lt, le = modelo(enc['input_ids'], enc['attention_mask'])
        seg = (time.time() - t0) / len(lote)
        for j, r in enumerate(lote):
            salidas.append({'polaridad': cfg['polaridad'][lp[j].argmax().item()],
                            'tema': cfg['temas'][lt[j].argmax().item()],
                            'entidades': decodificar_bio(le[j].argmax(-1).tolist(),
                                                         enc['offset_mapping'][j].tolist(), r['texto']),
                            'confianza_polaridad': lp[j].softmax(-1).max().item(),
                            'json_valido': True, 'segundos': seg})
    return salidas

> **Qué hace el código:** `etiquetas_bio` usa los `offsets` del tokenizador (posición de cada *token* en el texto) para asignar `B-`/`I-`/`O`; así funciona igual con BERT (WordPiece), RoBERTa (BPE) y XLM-R (SentencePiece). `decodificar_bio` hace el camino inverso: agrupa *tokens* consecutivos del mismo tipo y recupera el texto exacto de la entidad. Los tres encoders se entrenan con los mismos datos, hiperparámetros y semilla (**QA de la fase 3**).

In [ ]:
resultados_pred = {}   # (sector, modelo) -> lista de predicciones sobre el conjunto de prueba
modelos_encoder = {}
trains = {'comercial': train_com, 'juridico': train_jur}

for nombre_modelo, candidatos in ENCODERS.items():
    for sector in SECTORES:
        print(f'▶ {nombre_modelo} · {sector}: {len(trains[sector])} textos de entrenamiento')
        t0 = time.time()
        modelo, tok, repo = entrenar_encoder(candidatos, sector, trains[sector])
        resultados_pred[(sector, nombre_modelo)] = predecir_encoder(modelo, tok, sector, tests[sector])
        modelos_encoder[(sector, nombre_modelo)] = (modelo, tok, repo)
        print(f'  listo en {(time.time() - t0) / 60:.1f} min ({repo})')

> **Interpretación de la salida:** la pérdida debe bajar de una época a la siguiente; si sube o se estanca muy alta, el modelo no está aprendiendo (revisa las etiquetas). Anota los minutos de entrenamiento: forman parte del costo de cada opción, aunque se pagan una sola vez.

### 3.2 LLM con un solo *prompt* → JSON

El *prompt* define la tarea, las etiquetas permitidas y el formato de salida, e incluye **dos ejemplos del conjunto de entrenamiento** (*few-shot*), con sus etiquetas reales. La decodificación es voraz (sin muestreo) para que la salida sea reproducible.

**QA:** se registra si la respuesta es un JSON válido; si no lo es o trae una etiqueta fuera de la lista, la tarea cuenta como error (no se corrige a mano).

In [ ]:
def construir_prompt(sector: str, texto: str, ejemplos: list) -> list:
    cfg = SECTORES[sector]
    instrucciones = (
        f"Analiza el siguiente texto en español ({cfg['descripcion']}) y responde SOLO con un JSON con estas claves:\n"
        f"- \"polaridad\": {cfg['nombre_polaridad']}, uno de {cfg['polaridad']}\n"
        f"- \"tema\": uno de {cfg['temas']}\n"
        "- \"entidades\": lista de objetos {\"texto\": ..., \"tipo\": ...} copiando el texto exacto, con tipo en "
        "PER (persona), ORG (organización), LOC (lugar), FECHA (fecha) o MONTO (cantidad de dinero). "
        "Lista vacía si no hay.\n"
        "No agregues explicaciones."
    )
    mensajes = [{'role': 'system', 'content': 'Eres un analista de textos que responde únicamente en JSON válido.'}]
    for ej in ejemplos:
        respuesta = {'polaridad': ej['polaridad'], 'tema': ej['tema'],
                     'entidades': [{'texto': e['texto'], 'tipo': e['tipo']} for e in ej['entidades']]}
        mensajes += [{'role': 'user', 'content': f"{instrucciones}\n\nTexto: {ej['texto']}"},
                     {'role': 'assistant', 'content': json.dumps(respuesta, ensure_ascii=False)}]
    mensajes.append({'role': 'user', 'content': f"{instrucciones}\n\nTexto: {texto}"})
    return mensajes


def extraer_json(salida: str):
    """Primer objeto JSON balanceado de la salida del modelo; None si no hay uno válido."""
    inicio = salida.find('{')
    while inicio != -1:
        nivel = 0
        for k in range(inicio, len(salida)):
            nivel += {'{': 1, '}': -1}.get(salida[k], 0)
            if nivel == 0:
                fragmento = salida[inicio:k + 1]
                for intento in (fragmento, fragmento.replace("'", '"')):
                    try:
                        return json.loads(intento)
                    except json.JSONDecodeError:
                        pass
                break
        inicio = salida.find('{', inicio + 1)
    return None


def normalizar_etiqueta(valor, opciones: list):
    """Acepta mayúsculas y tildes distintas; cualquier otra cosa es inválida (None)."""
    v = sin_tildes(str(valor or '')).strip()
    for o in opciones:
        if v == sin_tildes(o):
            return o
    return None


def interpretar_salida(salida: str, sector: str) -> dict:
    cfg = SECTORES[sector]
    obj = extraer_json(salida)
    if not isinstance(obj, dict):
        return {'polaridad': None, 'tema': None, 'entidades': [], 'json_valido': False}
    ents = []
    for e in obj.get('entidades') or []:
        if isinstance(e, dict) and str(e.get('tipo', '')).upper() in TIPOS_ENT and str(e.get('texto', '')).strip():
            ents.append({'tipo': str(e['tipo']).upper(), 'texto': str(e['texto']).strip()})
    return {'polaridad': normalizar_etiqueta(obj.get('polaridad'), cfg['polaridad']),
            'tema': normalizar_etiqueta(obj.get('tema'), cfg['temas']),
            'entidades': ents, 'json_valido': True}


def elegir_ejemplos(train: list, n: int = 2) -> list:
    """Ejemplos few-shot del entrenamiento, con las tres etiquetas y de polaridades distintas."""
    completos = [r for r in train if r.get('polaridad') and r.get('tema') and r.get('entidades')]
    rng = random.Random(SEED)
    rng.shuffle(completos)
    elegidos, vistas = [], set()
    for r in completos:
        if r['polaridad'] not in vistas and len(r['texto'].split()) <= 120:
            elegidos.append(r); vistas.add(r['polaridad'])
        if len(elegidos) == n:
            break
    return elegidos


@torch.no_grad()
def predecir_llm(repo: str, sector: str, registros: list, ejemplos: list) -> list:
    tok = AutoTokenizer.from_pretrained(repo)
    try:   # transformers ≥ 4.56 usa 'dtype'; versiones anteriores, 'torch_dtype'
        modelo = AutoModelForCausalLM.from_pretrained(repo, dtype=torch.bfloat16)
    except TypeError:
        modelo = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16)
    modelo.eval()
    salidas = []
    for i, r in enumerate(registros):
        mensajes = construir_prompt(sector, r['texto'], ejemplos)
        try:
            ids = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors='pt')
        except Exception:  # noqa: BLE001  (plantillas sin rol 'system', p. ej. Gemma)
            mensajes = [m for m in mensajes if m['role'] != 'system']
            ids = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors='pt')
        if not isinstance(ids, torch.Tensor):          # versiones nuevas devuelven un diccionario
            ids = ids['input_ids']
        t0 = time.time()
        gen = modelo.generate(ids, max_new_tokens=220, do_sample=False,
                              pad_token_id=tok.pad_token_id or tok.eos_token_id)
        texto_salida = tok.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
        pred = interpretar_salida(texto_salida, sector)
        pred.update({'segundos': time.time() - t0, 'salida_cruda': texto_salida})
        salidas.append(pred)
        if (i + 1) % 10 == 0:
            print(f'    {i + 1}/{len(registros)} · {pred["segundos"]:.1f} s/texto')
    del modelo
    return salidas


# Prueba del intérprete con una salida típica (incluye texto extra antes del JSON)
print(interpretar_salida('Claro: {"polaridad": "Positivo", "tema": "hogar", '
                         '"entidades": [{"texto": "Bogotá", "tipo": "loc"}]}', 'comercial'))

> **Interpretación de la salida:** el intérprete tolera texto alrededor del JSON y diferencias de mayúsculas o tildes en las etiquetas (*Positivo* → *positivo*), pero no inventa respuestas: una etiqueta fuera de la lista queda en `None` y cuenta como error.

In [ ]:
for nombre_modelo, repo in LLMS.items():
    for sector in SECTORES:
        ejemplos = elegir_ejemplos(trains[sector])
        prueba = tests[sector][:N_PRUEBA_LLM]
        print(f'▶ {nombre_modelo} · {sector}: {len(prueba)} textos')
        resultados_pred[(sector, nombre_modelo)] = predecir_llm(repo, sector, prueba, ejemplos)

# Guarda las predicciones para no repetir la inferencia
with open(RESULTS_DIR / 'predicciones.json', 'w', encoding='utf-8') as f:
    json.dump({f'{s}|{m}': p for (s, m), p in resultados_pred.items()}, f, ensure_ascii=False, indent=1)
print('Predicciones guardadas en', RESULTS_DIR / 'predicciones.json')

> **Interpretación de la salida:** observa los segundos por texto: un LLM de 2B en CPU suele tardar varios segundos por texto, frente a fracciones de segundo de un encoder. Si un modelo tarda demasiado, reduce `N_PRUEBA_LLM` para clase y deja la corrida completa para fuera del aula. Revisa algunas `salida_cruda` en `predicciones.json`: muestran qué tan bien sigue el modelo el formato.

---

## 4. 🔷 Fase 4 — Evaluación

| Tarea | Métrica | Por qué |
|---|---|---|
| Polaridad | F1 macro | Las clases están desbalanceadas (sobre todo los fallos) |
| Tema | F1 macro | Igual peso a temas frecuentes y raros |
| Entidades | F1 micro por entidad: acierto si coinciden **texto y tipo** exactos | Estándar en NER; un acierto a medias cuenta como error |
| Conjunta | Puntaje global = promedio de los tres F1 | Resume el desempeño simultáneo en un número |
| Formato | % de JSON válido | Un LLM que no respeta el formato no es utilizable |
| Costo | Segundos por texto | Desempata entre modelos equivalentes |

**Comparación justa (QA):** todos los modelos se comparan sobre **los mismos textos**: los primeros `N_PRUEBA_LLM` del conjunto de prueba de cada sector (los que procesaron los LLM). Los encoders se reportan también sobre la prueba completa.

In [ ]:
def clave_entidad(e: dict) -> tuple:
    return (sin_tildes(re.sub(r'[\s.,;:]+$', '', e['texto'].strip())), e['tipo'])


def f1_entidades(golds: list, preds: list):
    vp = fp = fn = 0
    for g, p in zip(golds, preds):
        gs, ps = {clave_entidad(e) for e in g}, {clave_entidad(e) for e in p}
        vp += len(gs & ps); fp += len(ps - gs); fn += len(gs - ps)
    prec = vp / (vp + fp) if vp + fp else 0.0
    rec = vp / (vp + fn) if vp + fn else 0.0
    return (2 * prec * rec / (prec + rec) if prec + rec else 0.0), prec, rec


def evaluar(registros: list, preds: list) -> dict:
    gp = [r['polaridad'] for r in registros]; pp = [p['polaridad'] or 'INVALIDA' for p in preds]
    gt = [r['tema'] for r in registros]; pt = [p['tema'] or 'INVALIDA' for p in preds]
    f1p = f1_score(gp, pp, labels=sorted(set(gp)), average='macro', zero_division=0)
    f1t = f1_score(gt, pt, labels=sorted(set(gt)), average='macro', zero_division=0)
    f1e, pe, re_ = f1_entidades([r['entidades'] for r in registros], [p['entidades'] for p in preds])
    return {'F1 polaridad': f1p, 'F1 tema': f1t, 'F1 entidades': f1e,
            'Puntaje global': (f1p + f1t + f1e) / 3,
            'JSON válido (%)': 100 * np.mean([p.get('json_valido', True) for p in preds]),
            'Seg/texto': np.mean([p['segundos'] for p in preds]), 'n': len(registros)}


def linea_base(registros: list, train: list) -> list:
    """Predice siempre la clase más frecuente del entrenamiento y ninguna entidad."""
    mp = Counter(r['polaridad'] for r in train if r.get('polaridad')).most_common(1)[0][0]
    mt = Counter(r['tema'] for r in train if r.get('tema')).most_common(1)[0][0]
    return [{'polaridad': mp, 'tema': mt, 'entidades': [], 'segundos': 0.0} for _ in registros]


filas = []
for sector in SECTORES:
    sub = tests[sector][:N_PRUEBA_LLM]
    filas.append({'sector': sector, 'modelo': 'Línea base (mayoritaria)', 'familia': '—',
                  **evaluar(sub, linea_base(sub, trains[sector]))})
    for (s, m), preds in resultados_pred.items():
        if s == sector:
            familia = 'Encoder' if m in ENCODERS else 'LLM'
            filas.append({'sector': sector, 'modelo': m, 'familia': familia,
                          **evaluar(sub, preds[:len(sub)])})
tabla = pd.DataFrame(filas).sort_values(['sector', 'Puntaje global'], ascending=[True, False])
tabla.to_csv(RESULTS_DIR / 'comparacion_modelos.csv', index=False)
tabla.round(3)

> **Interpretación de la salida:** compara cada modelo con la **línea base** (clase mayoritaria y ninguna entidad): un modelo que no la supera claramente no aprendió la tarea. Lee la tabla por columnas: es común que los encoders ajustados ganen en entidades (aprenden el formato exacto de las etiquetas) y que los LLM pequeños rindan mejor en tema que en polaridad jurídica, o que pierdan puntos por JSON inválido. Las conclusiones dependen de tus datos validados; no las tomes de este texto.

In [ ]:
# Puntaje global promedio de los dos sectores y gráfico por tarea
resumen = (tabla[tabla['familia'] != '—'].groupby(['modelo', 'familia'])
           [['F1 polaridad', 'F1 tema', 'F1 entidades', 'Puntaje global', 'Seg/texto']].mean()
           .sort_values('Puntaje global', ascending=False))
print(resumen.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, sector in zip(axes, SECTORES):
    t = tabla[tabla['sector'] == sector].set_index('modelo')[['F1 polaridad', 'F1 tema', 'F1 entidades']]
    t.plot.barh(ax=ax, width=0.8)
    ax.set_title(f'Sector {sector}'); ax.set_xlim(0, 1); ax.set_xlabel('F1'); ax.invert_yaxis()
    ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

> **Interpretación de la salida:** el gráfico muestra si un modelo gana en todo o solo en una tarea. Un modelo con buen promedio pero F1 de entidades muy bajo no sirve si el caso de uso depende de extraer montos o entidades accionadas.

### 4.1 ¿La diferencia entre los dos mejores es real? Intervalo de confianza *bootstrap*

Con ~40–100 textos por sector, dos modelos pueden diferir por azar. Se remuestrean los textos de prueba con reemplazo 1.000 veces y se calcula la diferencia de puntaje global entre los dos mejores. Si el intervalo del 95 % incluye el 0, **no hay evidencia** de que uno sea mejor y se prefiere el más rápido (criterio de la fase 1).

In [ ]:
def bootstrap_diferencia(registros, preds_a, preds_b, n=1000):
    rng = np.random.default_rng(SEED)
    difs = []
    for _ in range(n):
        idx = rng.integers(0, len(registros), len(registros))
        r = [registros[i] for i in idx]
        difs.append(evaluar(r, [preds_a[i] for i in idx])['Puntaje global'] -
                    evaluar(r, [preds_b[i] for i in idx])['Puntaje global'])
    return np.percentile(difs, [2.5, 50, 97.5])


ganadores = {}
for sector in SECTORES:
    t = tabla[(tabla['sector'] == sector) & (tabla['familia'] != '—')]
    a, b = t.iloc[0]['modelo'], t.iloc[1]['modelo']
    sub = tests[sector][:N_PRUEBA_LLM]
    lo, med, hi = bootstrap_diferencia(sub, resultados_pred[(sector, a)][:len(sub)],
                                       resultados_pred[(sector, b)][:len(sub)])
    distinguible = lo > 0
    if not distinguible:   # empate estadístico: gana el más rápido
        a = t.iloc[:2].sort_values('Seg/texto').iloc[0]['modelo']
    ganadores[sector] = a
    print(f'{sector:10} {t.iloc[0]["modelo"]} − {t.iloc[1]["modelo"]}: diferencia={med:+.3f} '
          f'IC95%=[{lo:+.3f}, {hi:+.3f}] → {"diferencia real" if distinguible else "empate: se elige el más rápido"}'
          f' · elegido: {a}')

> **Interpretación de la salida:** un intervalo que cruza el 0 significa que, con estos datos, no se puede afirmar que el primero sea mejor que el segundo. Es frecuente con muestras pequeñas: para concluir con más seguridad, valida más textos o usa `MODO_RAPIDO = False`.

### 4.2 Análisis de errores del modelo elegido

In [ ]:
for sector, modelo in ganadores.items():
    sub = tests[sector][:N_PRUEBA_LLM]
    preds = resultados_pred[(sector, modelo)][:len(sub)]
    cfg = SECTORES[sector]
    print(f'=== {sector} · {modelo} ===')
    print(classification_report([r['polaridad'] for r in sub], [p['polaridad'] or 'INVALIDA' for p in preds],
                                labels=cfg['polaridad'], zero_division=0))
    # Errores de entidades por tipo
    por_tipo = {t: Counter() for t in TIPOS_ENT}
    for r, p in zip(sub, preds):
        gs, ps = {clave_entidad(e) for e in r['entidades']}, {clave_entidad(e) for e in p['entidades']}
        for _, t in gs & ps: por_tipo[t]['VP'] += 1
        for _, t in ps - gs: por_tipo[t]['FP'] += 1
        for _, t in gs - ps: por_tipo[t]['FN'] += 1
    print(pd.DataFrame(por_tipo).T.fillna(0).astype(int).to_string())
    # Tres ejemplos con la polaridad equivocada
    errores = [(r, p) for r, p in zip(sub, preds) if p['polaridad'] != r['polaridad']][:3]
    for r, p in errores:
        print(f"\n  real={r['polaridad']} · predicho={p['polaridad']} · {r['texto'][:180]}…")
    print()

> **Interpretación de la salida:** el reporte por clase muestra qué polaridad confunde el modelo (en el sector jurídico, *niega* e *improcedente* suelen mezclarse porque ambas son desfavorables al accionante). La tabla por tipo de entidad indica si fallan más los montos, las fechas o las organizaciones. Los ejemplos ayudan a decidir si el problema es del modelo o de la etiqueta: si la etiqueta está mal, corrígela en la hoja de validación (sección 2.6).

---

## 5. 🔷 Fase 5 — Despliegue

El modelo elegido en cada sector se expone con **una sola función**, `analizar(texto, sector)`, que devuelve siempre el mismo JSON, sea el modelo un encoder o un LLM. Se guarda además una **ficha del modelo** (IEEE 7001) con los datos, las métricas y los límites.

In [ ]:
_cache_llm = {}


def analizar(texto: str, sector: str) -> dict:
    """Polaridad, tema y entidades de un texto con el modelo elegido para el sector."""
    texto = truncar(texto)
    modelo = ganadores[sector]
    registro = {'texto': texto}
    if modelo in ENCODERS:
        enc, tok, _ = modelos_encoder[(sector, modelo)]
        pred = predecir_encoder(enc, tok, sector, [registro])[0]
    else:
        pred = predecir_llm(LLMS[modelo], sector, [registro], elegir_ejemplos(trains[sector]))[0]
    return {'sector': sector, 'modelo': modelo,
            SECTORES[sector]['nombre_polaridad']: pred['polaridad'], 'tema': pred['tema'],
            'entidades': pred['entidades'], 'json_valido': pred.get('json_valido', True)}


print(json.dumps(analizar('Compré una licuadora en la tienda de Medellín por $250.000 y llegó rota; '
                          'nadie responde los correos.', 'comercial'), ensure_ascii=False, indent=2))
print(json.dumps(analizar('La accionante solicitó a la Nueva EPS la entrega de un medicamento ordenado por su '
                          'médico tratante. La Sala amparó el derecho a la salud y ordenó a la EPS entregarlo '
                          'en 48 horas.', 'juridico'), ensure_ascii=False, indent=2))

> **Interpretación de la salida:** ambos ejemplos son **ficticios**. Verifica que el JSON tenga las tres respuestas y que las entidades sean fragmentos exactos del texto. Si el modelo elegido es un encoder, la respuesta es casi inmediata; si es un LLM, tarda varios segundos.

In [ ]:
# Guardar los encoders elegidos y la ficha del modelo
ficha = {'fecha': time.strftime('%Y-%m-%d'), 'proposito': 'Polaridad, tema y entidades simultáneos (uso académico)',
         'esquema': {s: {k: c[k] for k in ('polaridad', 'temas')} for s, c in SECTORES.items()} | {'entidades': TIPOS_ENT},
         'modo_rapido': MODO_RAPIDO, 'max_palabras': MAX_PALABRAS, 'modelos': {}}
for sector, modelo in ganadores.items():
    fila = tabla[(tabla['sector'] == sector) & (tabla['modelo'] == modelo)].iloc[0]
    info = {'modelo': modelo, 'metricas_prueba': {k: round(float(fila[k]), 4) for k in
            ['F1 polaridad', 'F1 tema', 'F1 entidades', 'Puntaje global', 'Seg/texto']},
            'textos_prueba': int(fila['n']), 'textos_validados': int(sum(r.get('validado', False) for r in tests[sector]))}
    if modelo in ENCODERS:
        enc, tok, repo = modelos_encoder[(sector, modelo)]
        carpeta = RESULTS_DIR / f'encoder_{sector}'
        carpeta.mkdir(exist_ok=True)
        torch.save(enc.state_dict(), carpeta / 'pesos.pt'); tok.save_pretrained(carpeta)
        info.update({'base': repo, 'carpeta': str(carpeta)})
    else:
        info['base'] = LLMS[modelo]
    ficha['modelos'][sector] = info
ficha['limitaciones'] = [
    'Entidades de entrenamiento y tema jurídico generados por reglas (etiquetas con ruido).',
    'MeLiSA: licencia no clara; uso solo académico.',
    'Textos truncados a las primeras palabras; sentencias representadas por su síntesis.',
    'No apto para decisiones judiciales ni de servicio al cliente sin revisión humana.',
]
(RESULTS_DIR / 'ficha_modelo.json').write_text(json.dumps(ficha, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(ficha['modelos'], ensure_ascii=False, indent=2))

> **Interpretación de la salida:** la ficha resume qué modelo se eligió por sector, con qué métricas y sobre cuántos textos validados. Es el documento que acompañaría al modelo si se entregara a otra área (IEEE 7001, ISO/IEC 42001).

---

## 6. 🔷 Fase 6 — Monitoreo y mantenimiento

Un modelo en uso se degrada si cambian los textos (nuevas categorías de producto, nueva jurisprudencia). Se proponen tres alarmas simples que se calculan sobre cada lote nuevo de textos, **sin necesidad de etiquetas**:

| Alarma | Qué se mide | Umbral sugerido |
|---|---|---|
| Deriva de predicciones | Distancia entre la distribución de polaridades y temas predichos y la del entrenamiento (índice de estabilidad poblacional, PSI) | PSI > 0.2 |
| Formato | % de respuestas sin JSON válido (LLM) | > 5 % |
| Confianza | Confianza media de la polaridad (encoder) | Caída > 10 puntos frente a la prueba |

Cuando una alarma se activa: se toma una muestra del lote, se valida en la misma hoja de la sección 2.6 y se vuelve a entrenar o a ajustar el *prompt*.

In [ ]:
def psi(referencia: list, actual: list, categorias: list, eps: float = 1e-4) -> float:
    """Índice de estabilidad poblacional entre dos distribuciones de etiquetas."""
    r = np.array([referencia.count(c) for c in categorias], dtype=float) / max(len(referencia), 1) + eps
    a = np.array([actual.count(c) for c in categorias], dtype=float) / max(len(actual), 1) + eps
    return float(np.sum((a - r) * np.log(a / r)))


def monitorear(sector: str, preds_lote: list) -> dict:
    cfg = SECTORES[sector]
    ref_pol = [r['polaridad'] for r in trains[sector] if r.get('polaridad')]
    ref_tema = [r['tema'] for r in trains[sector] if r.get('tema')]
    informe = {'PSI polaridad': psi(ref_pol, [p['polaridad'] for p in preds_lote], cfg['polaridad']),
               'PSI tema': psi(ref_tema, [p['tema'] for p in preds_lote], cfg['temas']),
               'JSON inválido (%)': float(100 * (1 - np.mean([p.get('json_valido', True) for p in preds_lote])))}
    informe['alarma'] = (informe['PSI polaridad'] > 0.2 or informe['PSI tema'] > 0.2
                         or informe['JSON inválido (%)'] > 5)
    return informe


# Simulación: el conjunto de prueba hace las veces de «lote nuevo»
for sector, modelo in ganadores.items():
    print(sector, {k: (round(v, 3) if isinstance(v, float) else v)
                   for k, v in monitorear(sector, resultados_pred[(sector, modelo)]).items()})

> **Interpretación de la salida:** con el conjunto de prueba, que viene de la misma fuente que el entrenamiento, el PSI debería ser bajo (< 0.1) y no debería saltar ninguna alarma. En uso real, un PSI alto indica que llegan textos distintos (por ejemplo, una avalancha de tutelas de un mismo tema) y que conviene revisar una muestra.

---

## Conclusiones

Completa esta sección **después** de ejecutar el notebook con el conjunto de prueba validado:

| Pregunta | Respuesta (según tus resultados) |
|---|---|
| ¿Qué modelo obtuvo el mayor puntaje global en cada sector? | |
| ¿La diferencia con el segundo es estadísticamente real (sección 4.1)? | |
| ¿En qué tarea fallan más los LLM pequeños? ¿Y los encoders? | |
| ¿Cuánto más lento es el mejor LLM que el mejor encoder? | |
| ¿Qué etiquetas automáticas hubo que corregir más en la validación? | |

**Preguntas para discutir en clase:**

1. ¿Vale la pena un LLM si un encoder de 110 M parámetros logra un puntaje parecido en una fracción del tiempo?
2. ¿Qué cambiaría si en lugar de 2 ejemplos *few-shot* se usaran 5? ¿Y si se ajustara el LLM (LoRA)?
3. ¿Qué riesgos tiene usar este sistema para priorizar tutelas? ¿Qué controles humanos exigirías (IEEE 7000)?

## Referencias

- Studer, S. et al. (2021). Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology. *Machine Learning and Knowledge Extraction*, 3(2).
- Cañete, J. et al. (2020). Spanish Pre-Trained BERT Model and Evaluation Data (BETO). PML4DC, ICLR.
- Gutiérrez-Fandiño, A. et al. (2022). MarIA: Spanish Language Models. *Procesamiento del Lenguaje Natural*, 68.
- Conneau, A. et al. (2020). Unsupervised Cross-lingual Representation Learning at Scale (XLM-R). ACL.
- Language Technologies Unit, Barcelona Supercomputing Center (2024). *Salamandra* — ficha del modelo e informe técnico.
- Qwen Team (2024). Qwen2.5 Technical Report.
- MeLiSA: Mercado Libre for Sentiment Analysis, LPSC-FIUBA. NextProcurement NER, BSC. Relatoría de la Corte Constitucional de Colombia.